In [1]:
import pandas as pd
import numpy as np

# Load your cleaned data
df = pd.read_csv('Cleaned_Online_Retail.csv')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [2]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


### Transactional & Order Level Features

---

In [3]:
df1 = df.copy()
df1['LineTotal'] = df1['Quantity'] * df1['UnitPrice']
df1['IsCancellation'] = df1['InvoiceNo'].astype(str).str.startswith('C')

df1.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,LineTotal,IsCancellation
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,False
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,False
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,False
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,False
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,False


In [6]:
# Basket features (Grouped by InvoiceNo)
invoice_metrics = df1.groupby('InvoiceNo').agg(
    BasketSize=('Quantity', 'sum'),
    BasketValue=('LineTotal', 'sum'),
    UniqueItemsCount=('StockCode', 'nunique')
).reset_index()

df1 = df1.merge(invoice_metrics, on='InvoiceNo', how='left')

In [8]:
df1.head(3)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,LineTotal,IsCancellation,BasketSize,BasketValue,UniqueItemsCount
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,False,40,139.12,7
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,False,40,139.12,7
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,False,40,139.12,7


### Temporal Features

---

In [9]:
df1['OrderHour'] = df1['InvoiceDate'].dt.hour
df1['DayOfWeek'] = df1['InvoiceDate'].dt.day_name()
df1['IsWeekend'] = df1['InvoiceDate'].dt.dayofweek.isin([5, 6]).astype(int)
df1['OrderMonth'] = df1['InvoiceDate'].dt.month
df1['OrderQuarter'] = df1['InvoiceDate'].dt.quarter

In [10]:
# Bins for TimeOfDay
bins = [0, 6, 12, 17, 21, 24]
labels = ['Night', 'Morning', 'Afternoon', 'Evening', 'Night']
df1['TimeOfDay'] = pd.cut(df1['OrderHour'], bins=bins, labels=labels, ordered=False)

In [11]:
df1.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,LineTotal,IsCancellation,BasketSize,BasketValue,UniqueItemsCount,OrderHour,DayOfWeek,IsWeekend,OrderMonth,OrderQuarter,TimeOfDay
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30,False,40,139.12,7,8,Wednesday,0,12,4,Morning
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,False,40,139.12,7,8,Wednesday,0,12,4,Morning
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00,False,40,139.12,7,8,Wednesday,0,12,4,Morning
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,False,40,139.12,7,8,Wednesday,0,12,4,Morning
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34,False,40,139.12,7,8,Wednesday,0,12,4,Morning


### Customer Level Features

---

In [13]:
# Filter out cancellations for monetary metrics
rfm_df = df1[~df1['IsCancellation']].copy()
max_date = rfm_df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = rfm_df.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (max_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('LineTotal', 'sum')
).reset_index()

In [14]:
# Scoring RFM (Quantiles 1-5)
for col in ['Recency', 'Frequency', 'Monetary']:
    # Note: For Recency, lower is better, so we reverse the score
    if col == 'Recency':
        rfm[f'{col}Score'] = pd.qcut(rfm[col], 5, labels=[5, 4, 3, 2, 1])
    else:
        rfm[f'{col}Score'] = pd.qcut(rfm[col].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])

In [15]:
# Create combined RFM Score
rfm['RFM_Segment'] = rfm['RecencyScore'].astype(str) + rfm['FrequencyScore'].astype(str) + rfm['MonetaryScore'].astype(str)

In [16]:
rfm.head()

,CustomerID,Recency,Frequency,Monetary,RecencyScore,FrequencyScore,MonetaryScore,RFM_Segment
0,12346.0,326,1,77183.60,1,1,5,115
1,12347.0,2,7,4310.00,5,5,5,555
2,12348.0,75,4,1797.24,2,4,4,244
3,12349.0,19,1,1757.55,4,1,4,414
4,12350.0,310,1,334.40,1,1,2,112


### Save the dataset

---

In [17]:
df1.to_csv('Feature_Engineered_Online_Retail.csv', index=False)
rfm.to_csv('customer_level_features.csv', index=False)